# AI_SENTIMENT — Product Review Analysis

`AI_SENTIMENT` returns a sentiment score from **-1** (negative) to **+1** (positive).

| Item | Detail |
|---|---|
| **Function** | `SNOWFLAKE.CORTEX.AI_SENTIMENT` |
| **Data** | 30 synthetic product reviews (inline) |
| **Use-case** | Automated sentiment scoring for customer feedback |


## Step 1 — Environment & Table Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

In [ ]:
CREATE OR REPLACE TABLE product_reviews (
    review_id       INT AUTOINCREMENT,
    product_name    VARCHAR(200),
    category        VARCHAR(100),
    review_text     TEXT,
    star_rating     FLOAT,
    created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

In [ ]:
INSERT INTO product_reviews (product_name, category, review_text, star_rating) VALUES
('Wireless Headphones Pro', 'Electronics', 'Absolutely love these headphones! Crystal clear sound and the noise cancellation is incredible. Best purchase I have made this year.', 5.0),
('Wireless Headphones Pro', 'Electronics', 'Sound quality is decent but they hurt my ears after an hour. The padding needs improvement.', 3.0),
('Wireless Headphones Pro', 'Electronics', 'Terrible battery life. Dies after 2 hours despite claiming 20 hours. Complete false advertising.', 1.0),
('Running Shoes Ultra', 'Sports', 'These shoes transformed my running experience. Super lightweight and great arch support.', 5.0),
('Running Shoes Ultra', 'Sports', 'Good shoes but the sizing runs small. Had to exchange for a larger pair.', 3.0),
('Running Shoes Ultra', 'Sports', 'Sole came apart after just two weeks of use. Very disappointing quality.', 1.0),
('Smart Home Hub', 'Electronics', 'Setup was a breeze and it works perfectly with all my devices. Voice control is very responsive.', 5.0),
('Smart Home Hub', 'Electronics', 'Works okay but the app is clunky and crashes frequently on Android.', 2.0),
('Smart Home Hub', 'Electronics', 'Constantly loses connection to WiFi. Had to restart it multiple times daily.', 1.0),
('Organic Face Cream', 'Beauty', 'My skin has never looked better! The natural ingredients really make a difference.', 5.0),
('Organic Face Cream', 'Beauty', 'Nice texture but did not see any dramatic results after a month of use.', 3.0),
('Organic Face Cream', 'Beauty', 'Caused a severe allergic reaction. The ingredient list is misleading.', 1.0),
('Standing Desk Pro', 'Furniture', 'Rock solid build quality and the electric motor is whisper quiet. Worth every penny.', 5.0),
('Standing Desk Pro', 'Furniture', 'Desk is good but assembly instructions were confusing. Took 3 hours.', 3.0),
('Standing Desk Pro', 'Furniture', 'The motor stopped working after a month. Customer service has been unhelpful.', 1.0),
('Python Cookbook', 'Books', 'Excellent resource for intermediate Python developers. Well-structured examples and clear explanations.', 5.0),
('Python Cookbook', 'Books', 'Some chapters are great but others feel rushed. Could use more real-world examples.', 3.0),
('Python Cookbook', 'Books', 'Full of typos and the code examples do not even run. Clearly not reviewed properly.', 1.0),
('Yoga Mat Premium', 'Sports', 'Perfect thickness and grip. No sliding during hot yoga sessions anymore.', 5.0),
('Yoga Mat Premium', 'Sports', 'Mat is okay but it has a strong chemical smell that has not gone away after weeks.', 2.0),
('Espresso Machine Deluxe', 'Kitchen', 'Makes cafe-quality espresso at home. The built-in grinder is a game changer.', 5.0),
('Espresso Machine Deluxe', 'Kitchen', 'Good espresso but the steam wand is weak. Takes forever to froth milk.', 3.0),
('Espresso Machine Deluxe', 'Kitchen', 'Leaked water everywhere on first use. Returned immediately.', 1.0),
('Bluetooth Speaker Mini', 'Electronics', 'Surprisingly powerful sound for such a small speaker. Great for travel.', 4.0),
('Bluetooth Speaker Mini', 'Electronics', 'Average sound quality. The bass is almost nonexistent.', 2.0),
('Travel Backpack 40L', 'Travel', 'Perfect carry-on size with thoughtful compartments. Used it for a 2-week Europe trip.', 5.0),
('Travel Backpack 40L', 'Travel', 'Zippers feel cheap and one already broke. Not worth the premium price.', 2.0),
('Instant Pot 7-in-1', 'Kitchen', 'Replaced four kitchen appliances. The pressure cooking feature saves so much time.', 5.0),
('Instant Pot 7-in-1', 'Kitchen', 'Works well but the seal ring absorbs odors. Need separate rings for different foods.', 3.0),
('Instant Pot 7-in-1', 'Kitchen', 'Display stopped working after 3 months. Extremely frustrating for an expensive appliance.', 1.0);

In [ ]:
SELECT * FROM product_reviews LIMIT 5;

## Step 2 — Sentiment Analysis

Apply `AI_SENTIMENT` to each review and compare with the star rating.

In [ ]:
SELECT
    review_id,
    product_name,
    star_rating,
    ROUND(SNOWFLAKE.CORTEX.AI_SENTIMENT(review_text), 3) AS sentiment_score,
    LEFT(review_text, 80) AS review_snippet
FROM product_reviews
ORDER BY sentiment_score DESC
LIMIT 15;

## Step 3 — Sentiment by Product & Category

In [ ]:
SELECT
    product_name,
    COUNT(*)                                                    AS review_count,
    ROUND(AVG(star_rating), 2)                                  AS avg_stars,
    ROUND(AVG(SNOWFLAKE.CORTEX.AI_SENTIMENT(review_text)), 3)   AS avg_sentiment
FROM product_reviews
GROUP BY product_name
ORDER BY avg_sentiment DESC;

In [ ]:
SELECT
    category,
    COUNT(*)                                                    AS review_count,
    ROUND(AVG(star_rating), 2)                                  AS avg_stars,
    ROUND(AVG(SNOWFLAKE.CORTEX.AI_SENTIMENT(review_text)), 3)   AS avg_sentiment
FROM product_reviews
GROUP BY category
ORDER BY avg_sentiment DESC;

## Step 4 — Find Star-Sentiment Mismatches

In [ ]:
SELECT
    product_name,
    star_rating,
    ROUND(SNOWFLAKE.CORTEX.AI_SENTIMENT(review_text), 3) AS sentiment_score,
    review_text
FROM product_reviews
WHERE ABS(star_rating / 5.0 - (SNOWFLAKE.CORTEX.AI_SENTIMENT(review_text) + 1) / 2) > 0.3
ORDER BY star_rating DESC;

## Key Takeaways

- `AI_SENTIMENT` needs no training — works out of the box on any text.
- Returns a continuous score (-1 to +1), useful for ranking and thresholding.
- Compare AI sentiment vs. star ratings to find gaming or mismatched feedback.
- Aggregate by product/category for instant voice-of-customer dashboards.
